In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Fantasy value will be decided by ESPN fantasy scoring metrics. Rebounds & Points = 1 fantasy pt, 3-pointer = 1 pt, FGM = 2 pts, FTM = 1 pt, Assists = 2 pts, Steals & Blocks = 4 pts, FGA = -1 pt, FTA = -1 pt, & turnovers = -2 pts.")

# Load and clean data
df = pd.read_csv("Player_Totals.csv", index_col=0)
df.columns = df.columns.str.strip().str.lower()
df = df[df['season'] >= 2020]
#Prevent skewing in plots
df = df[df['pts'] >= 300]
#df = df[df['fga'] >= 250]
#df = df[df['fta'] >= 120]
columns_drop = ["orb", "drb", "mp", "gs", "birth_year", "lg"]
df = df.drop(columns=columns_drop, errors='ignore')

In [ ]:
#Define fantasy point calculation function
def calc_fantasy_points(row):
    twopm = row['fg'] - row['x3p']
    return (
        row['x3p'] * 5 +
        twopm * 3 +
        row['ft'] * 1 +
        (row['fga'] + row['fta']) * -1 +
        row['trb'] * 1 +
        row['ast'] * 2 +
        row['stl'] * 4 +
        row['blk'] * 4 +
        row['tov'] * -2
    )

# Apply fantasy points calculation
df['fantasy_points'] = df.apply(calc_fantasy_points, axis=1)
df['fantasy_ppg'] = df['fantasy_points'] / df['g']
df['season_weight'] = 2.5 ** (df['season'] - 2020)
df['weighted_fp'] = df['fantasy_ppg'] * df['season_weight']


In [3]:
# Compute weighted fantasy stats
player_weighted = df.groupby('player').agg({
    'weighted_fp': 'sum',
    'season_weight': 'sum'
}).reset_index()
player_weighted['weighted_fantasy_ppg'] = player_weighted['weighted_fp'] / player_weighted['season_weight']

# Durability calculation
df_recent = df[df['season'].isin([2024, 2025])]
recent_games = df_recent.groupby('player')['g'].sum().reset_index()
recent_games.rename(columns={'g': 'games_last_two_seasons'}, inplace=True)


# Aggregate per-player stats
df_fantasy = df.groupby('player').agg({
    'fantasy_ppg': 'mean',
    'fantasy_points': 'sum',
    'pts': 'mean',
    'trb': 'mean',
    'ast': 'mean',
    'stl': 'mean',
    'blk': 'mean',
    'tov': 'mean',
    'fg_percent': 'mean',
    'x3p_percent': 'mean',
    'x2p_percent': 'mean',
    'e_fg_percent': 'mean',
    'g': 'sum',
    'season': 'count'
}).rename(columns={'season': 'num_seasons'}).reset_index()
# Merge extra info
df_fantasy = df_fantasy.merge(recent_games, on='player', how='left')
df_fantasy['games_last_two_seasons'] = df_fantasy['games_last_two_seasons'].fillna(0)
df_fantasy['durability'] = pd.cut(
    df_fantasy['games_last_two_seasons'],
    bins=[-1, 40, 100, 140, float('inf')],
    labels=['Very Risky', 'Risky', 'Moderate', 'Reliable']
)



In [4]:


player_pos = df[['player', 'pos']].drop_duplicates(subset='player')
df_fantasy = df_fantasy.merge(player_pos, on='player', how='left')
df_fantasy = df_fantasy.merge(player_weighted[['player', 'weighted_fantasy_ppg']], on='player', how='left')

#Defining high-risk injury players
major_injury_flags = {
    'Tyrese Haliburton': 'Hamstring - Reinjury Risk / Achilles Tear',
    'Zion Williamson': 'Lower Body / Conditioning',
    'Lonzo Ball': 'Cartilage Replacement Surgery',
    'Jamal Murray': 'ACL Recovery',
    'Kristaps Porziņģis': 'Achilles / Calf Strain',
    'Kawhi Leonard': 'Chronic Knee (Degeneration)',
    'Paul George': 'Multiple Lower Body Injuries',
    'Ja Morant': 'Shoulder Surgery + Off-court Issues',
    'Anthony Davis': 'Recurring Foot / Back Problems',
    'Michael Porter Jr.': 'Back Injury History'
}

#Adding injury note column to fantasy dataframe
df_fantasy['injury_note'] = df_fantasy['player'].map(major_injury_flags).fillna('')


# Final formatting
cols = ['player', 'weighted_fantasy_ppg'] + [col for col in df_fantasy.columns if col not in ['player', 'weighted_fantasy_ppg']]
df_fantasy = df_fantasy[cols]
df_fantasy = df_fantasy[df_fantasy['g'] >= 120]
df_fantasy = df_fantasy.sort_values(by='weighted_fantasy_ppg', ascending=False)

In [ ]:
while True:
    print("\n NBA Fantasy Console Menu")
    print("1. Show Top 100 Fantasy Players (Weighted Fantasy PPG)")
    print("2. Show Top 20 Fantasy Players by Position")
    print("3. Show Advanced Metrics for a Specific Player")
    print("4. Exit")

    choice = input("Enter your choice (1–4): ").strip()

    if choice == '1':
        top100 = df_fantasy.head(100)
        print("\nTop 100 NBA Fantasy Players:")
        print(top100[['player', 'pos', 'weighted_fantasy_ppg', 'fantasy_ppg', 'durability', 'injury_note']])

    elif choice == '2':
        pos_input = input("Enter a position (e.g., PG, SG, SF, PF, C): ").strip().upper()
        top_pos = df_fantasy[df_fantasy['pos'] == pos_input].head(20)
        if top_pos.empty:
            print(f"No players found for position '{pos_input}'.")
        else:
            print(f"\nTop 20 {pos_input}s by Weighted Fantasy PPG:")
            print(top_pos[['player', 'pos', 'weighted_fantasy_ppg', 'fantasy_ppg', 'durability','injury_note']])

    elif choice == '3':
        name_input = input("Enter the full or partial name of the player: ").strip().lower()
        matches = df[df['player'].str.lower().str.contains(name_input)]

        if matches.empty:
            print(f"No players found matching '{name_input}'.")
        else:
            selected_player = matches.iloc[0]['player']
            player_row = matches[matches['player'] == selected_player]

            print(f"\nAdvanced Stats for {selected_player} (per game averages shown, 2020-2025):")
            # Show injury warning if available
            injury_flag = df_fantasy[df_fantasy['player'] == selected_player]['injury_note'].values
            if injury_flag.size > 0 and injury_flag[0]:
                print(f"Injury Note: {injury_flag[0]}")

            player_per_game = {
                'PPG': (player_row['pts'] / player_row['g']).mean(),
                'RPG': (player_row['trb'] / player_row['g']).mean(),
                'APG': (player_row['ast'] / player_row['g']).mean(),
                'SPG': (player_row['stl'] / player_row['g']).mean(),
                'BPG': (player_row['blk'] / player_row['g']).mean(),
                'TOPG': (player_row['tov'] / player_row['g']).mean(),
                'FG%': player_row['fg_percent'].mean(),
                '3P%': (player_row['x3p_percent'] * 100).mean(),
                '2P%': player_row['x2p_percent'].mean(),
                'eFG%': player_row['e_fg_percent'].mean()
            }

            for stat, val in player_per_game.items():
                print(f"{stat}: {val:.2f}")

            stat_mapping = {
                'PPG': df['pts'] / df['g'],
                'RPG': df['trb'] / df['g'],
                'APG': df['ast'] / df['g'],
                'SPG': df['stl'] / df['g'],
                'BPG': df['blk'] / df['g'],
                'TOPG': df['tov'] / df['g'],
                'FG%': df['fg_percent'],
                '3P%': df['x3p_percent'] * 100,
                '2P%': df['x2p_percent'],
                'eFG%': df['e_fg_percent']
            }

            fig, axes = plt.subplots(4, 3, figsize=(18, 16))
            axes = axes.flatten()

            for i, (stat_name, _) in enumerate(stat_mapping.items()):
                if stat_name == '3P%':
                    mask = df['x3pa'] >= 150
                    plot_series = df.loc[mask, 'x3p_percent'] * 100
                    title = f"{stat_name} Distribution (min 100 3PA)"

                elif stat_name == '2P%':
                    mask = df['x2pa'] >= 200
                    plot_series = df.loc[mask, 'x2p_percent'] 
                    title = f"{stat_name} Distribution (min 200 2PA)"

                elif stat_name == 'PPG':
                    plot_series = df['pts'] / df['g']
                    title = f"{stat_name} Distribution"

                elif stat_name == 'RPG':
                    plot_series = df['trb'] / df['g']
                    title = f"{stat_name} Distribution"

                elif stat_name == 'APG':
                    plot_series = df['ast'] / df['g']
                    title = f"{stat_name} Distribution"

                elif stat_name == 'SPG':
                    plot_series = df['stl'] / df['g']
                    title = f"{stat_name} Distribution"

                elif stat_name == 'BPG':
                    plot_series = df['blk'] / df['g']
                    title = f"{stat_name} Distribution"

                elif stat_name == 'TOPG':
                    plot_series = df['tov'] / df['g']
                    title = f"{stat_name} Distribution"

                elif stat_name == 'FG%':
                    plot_series = df['fg_percent']
                    title = f"{stat_name} Distribution"

                elif stat_name == 'eFG%':
                    plot_series = df['e_fg_percent']
                    title = f"{stat_name} Distribution"

                else:
                    continue  # just in case

            
                sns.histplot(plot_series, kde=True, ax=axes[i], color='skyblue')
                axes[i].axvline(player_per_game[stat_name], color='red', linestyle='--', label=selected_player)
                axes[i].set_title(title)
                axes[i].legend()

            for j in range(i + 1, len(axes)):
                axes[j].set_visible(False)





        plt.tight_layout()
        plt.show()



    elif choice == '4':
        print("Exiting NBA Fantasy Console. Have a great season!")
        break

    else:
        print("Invalid choice. Please enter 1, 2, 3, or 4.")



 NBA Fantasy Console Menu
1. Show Top 100 Fantasy Players (Weighted Fantasy PPG)
2. Show Top 20 Fantasy Players by Position
3. Show Advanced Metrics for a Specific Player
4. Exit

Top 20 Cs by Weighted Fantasy PPG:
                 player pos  weighted_fantasy_ppg  fantasy_ppg durability  \
411        Nikola Jokić   C             50.686561    45.181881   Reliable   
136    Domantas Sabonis   C             35.662769    33.623177   Reliable   
413      Nikola Vučević   C             30.999608    32.395877   Reliable   
301  Karl-Anthony Towns   C             30.749260    31.268154   Moderate   
266         Joel Embiid   C             29.681026    31.921926      Risky   
32          Bam Adebayo   C             29.031529    29.046369   Reliable   
14       Alperen Şengün   C             28.476655    25.414384   Moderate   
532      Walker Kessler   C             28.277903    26.406148   Moderate   
332  Kristaps Porziņģis   C             28.091491    28.259340      Risky   
457         Ru